# 04 — Player Projections

This notebook builds player level projections that will eventually feed into the team strength model.

The goal is to estimate how much value each player and position group should provide in the upcoming season using recent performance, playing time, age, experience, injuries, roster context, and regression toward league average.

Quarterback receives additional attention because of its outsized impact on team performance.

The outputs from this notebook will later be combined with roster movement and team level features to create projected 2026 team strength.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import nflreadpy as nfl

from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

## Load Player Data

The projection process starts with historical player production, playing time, roster information, and team context.

I keep the initial inputs limited to the datasets needed to build the player season modeling table, then add other sources only when they provide a specific projection feature.

In [3]:
player_stats = pl.read_parquet(
    PROCESSED_DIR / "player_stats_clean.parquet"
)

snap_counts = pl.read_parquet(
    PROCESSED_DIR / "snap_counts_clean.parquet"
)

rosters = pl.read_parquet(
    PROCESSED_DIR / "rosters_clean.parquet"
)

injuries = pl.read_parquet(
    PROCESSED_DIR / "injuries_clean.parquet"
)

draft_picks_clean = pl.read_parquet(
    PROCESSED_DIR / "draft_picks_clean.parquet"
)

In [4]:
for name, df in [
    ("player_stats", player_stats),
    ("snap_counts", snap_counts),
    ("rosters", rosters),
    ("injuries", injuries)
]:
    print(f"\n{name}")
    print("Shape:", df.shape)
    print(
        "Seasons:",
        df["season"].min(),
        "to",
        df["season"].max()
    )
    print(df.columns)


player_stats
Shape: (21366, 148)
Seasons: 2015 to 2025
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'season_type', 'recent_team', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'rec

## Build the Historical Player Season Base

The projection models need one consistent historical row for each player season.

Player statistics use GSIS IDs while snap counts use PFR IDs, so I use the roster data as the bridge between the two systems. Before combining the datasets, I validate that the identifiers and player season records are unique enough for reliable joins.

In [5]:
print(
    "Duplicate player_stats player-seasons:",
    player_stats
    .group_by([
        "season",
        "player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(
    "Roster GSIS mappings with multiple PFR IDs:",
    rosters
    .filter(
        pl.col("gsis_id").is_not_null() &
        pl.col("pfr_id").is_not_null()
    )
    .group_by([
        "season",
        "gsis_id"
    ])
    .agg(
        pl.col("pfr_id").n_unique().alias("pfr_ids")
    )
    .filter(pl.col("pfr_ids") > 1)
    .height
)

print(
    "Roster rows per player-season > 1:",
    rosters
    .filter(
        pl.col("gsis_id").is_not_null()
    )
    .group_by([
        "season",
        "gsis_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(
    "Snap PFR player-seasons:",
    snap_counts
    .group_by([
        "season",
        "pfr_player_id"
    ])
    .len()
    .height
)

Duplicate player_stats player-seasons: 0
Roster GSIS mappings with multiple PFR IDs: 0
Roster rows per player-season > 1: 114
Snap PFR player-seasons: 23220


### Player Identifier Bridge

Player statistics identify players with GSIS IDs, while snap count data uses PFR IDs.

The roster dataset provides a reliable bridge between these identifier systems. Because some players appear on multiple roster records within the same season, I collapse roster information to one player season while preserving the stable player attributes needed for projection modeling.

In [6]:
roster_bridge = (
    rosters
    .filter(
        pl.col("gsis_id").is_not_null()
    )
    .sort([
        "season",
        "gsis_id",
        "team"
    ])
    .group_by([
        "season",
        "gsis_id"
    ])
    .agg([
        pl.col("pfr_id")
        .drop_nulls()
        .first()
        .alias("pfr_id"),

        pl.col("full_name")
        .drop_nulls()
        .first()
        .alias("roster_name"),

        pl.col("position")
        .drop_nulls()
        .first()
        .alias("roster_position"),

        pl.col("years_exp")
        .drop_nulls()
        .max()
        .alias("years_exp"),

        pl.col("birth_date")
        .drop_nulls()
        .first()
        .alias("birth_date"),

        pl.col("height")
        .drop_nulls()
        .first()
        .alias("height"),

        pl.col("weight")
        .drop_nulls()
        .first()
        .alias("weight"),

        pl.col("college")
        .drop_nulls()
        .first()
        .alias("college"),

        pl.col("team")
        .drop_nulls()
        .unique()
        .sort()
        .alias("roster_teams"),

        pl.col("team")
        .drop_nulls()
        .n_unique()
        .alias("roster_team_count")
    ])
)

In [7]:
player_season_snaps = (
    snap_counts
    .filter(
        pl.col("pfr_player_id").is_not_null()
    )
    .group_by([
        "season",
        "pfr_player_id"
    ])
    .agg([
        pl.col("offense_snaps")
        .fill_null(0)
        .sum()
        .alias("offense_snaps"),

        pl.col("defense_snaps")
        .fill_null(0)
        .sum()
        .alias("defense_snaps"),

        pl.col("st_snaps")
        .fill_null(0)
        .sum()
        .alias("special_teams_snaps"),

        pl.col("game_id")
        .n_unique()
        .alias("games_with_snaps"),

        pl.col("team")
        .drop_nulls()
        .unique()
        .sort()
        .alias("snap_teams")
    ])
    .with_columns(
        (
            pl.col("offense_snaps")
            + pl.col("defense_snaps")
            + pl.col("special_teams_snaps")
        ).alias("total_snaps")
    )
)

In [8]:
player_seasons = (
    player_stats
    .join(
        roster_bridge,
        left_on=["season", "player_id"],
        right_on=["season", "gsis_id"],
        how="left"
    )
    .join(
        player_season_snaps,
        left_on=["season", "pfr_id"],
        right_on=["season", "pfr_player_id"],
        how="left"
    )
)

In [9]:
snap_coverage_by_season = (
    player_seasons
    .group_by("season")
    .agg([
        pl.len().alias("player_seasons"),

        pl.col("pfr_id")
        .is_not_null()
        .sum()
        .alias("with_pfr_id"),

        pl.col("total_snaps")
        .is_not_null()
        .sum()
        .alias("with_snap_data")
    ])
    .with_columns([
        (
            pl.col("with_pfr_id")
            / pl.col("player_seasons")
        ).alias("pfr_coverage"),

        (
            pl.col("with_snap_data")
            / pl.col("player_seasons")
        ).alias("snap_coverage")
    ])
    .sort("season")
)

snap_coverage_by_season

season,player_seasons,with_pfr_id,with_snap_data,pfr_coverage,snap_coverage
i32,u32,u32,u32,f64,f64
2015,1845,799,798,0.433062,0.43252
2016,1855,921,921,0.496496,0.496496
2017,1868,1040,1039,0.556745,0.55621
2018,1883,1155,1155,0.613383,0.613383
2019,1888,1173,1173,0.621292,0.621292
…,…,…,…,…,…
2021,2081,1302,1302,0.625661,0.625661
2022,2006,1081,1081,0.538883,0.538883
2023,1942,1427,1425,0.734809,0.73378


In [10]:
snap_coverage_by_position = (
    player_seasons
    .filter(
        pl.col("position").is_in([
            "QB",
            "RB",
            "WR",
            "TE"
        ])
    )
    .group_by("position")
    .agg([
        pl.len().alias("player_seasons"),

        pl.col("pfr_id")
        .is_not_null()
        .sum()
        .alias("with_pfr_id"),

        pl.col("total_snaps")
        .is_not_null()
        .sum()
        .alias("with_snap_data")
    ])
    .with_columns([
        (
            pl.col("with_pfr_id")
            / pl.col("player_seasons")
        ).alias("pfr_coverage"),

        (
            pl.col("with_snap_data")
            / pl.col("player_seasons")
        ).alias("snap_coverage")
    ])
    .sort("position")
)

snap_coverage_by_position

position,player_seasons,with_pfr_id,with_snap_data,pfr_coverage,snap_coverage
str,u32,u32,u32,f64,f64
"""QB""",847,749,749,0.884298,0.884298
"""RB""",1667,1344,1344,0.806239,0.806239
"""TE""",1396,1152,1151,0.825215,0.824499
"""WR""",2523,2059,2056,0.816092,0.814903


## Standardize Projection Positions

Player projections differ substantially by position because production, playing time, aging, and team impact are not comparable across roles.

I standardize players into projection groups before creating historical performance features. Quarterbacks are kept separate because of their outsized impact on team performance, while offensive and defensive players are grouped by their primary role.

In [11]:
player_seasons = (
    player_seasons
    .with_columns(
        pl.when(pl.col("position") == "QB")
        .then(pl.lit("QB"))

        .when(pl.col("position").is_in(["RB", "FB"]))
        .then(pl.lit("RB"))

        .when(pl.col("position") == "WR")
        .then(pl.lit("WR"))

        .when(pl.col("position") == "TE")
        .then(pl.lit("TE"))

        .when(pl.col("position").is_in([
            "T", "OT", "G", "OG", "C", "OL"
        ]))
        .then(pl.lit("OL"))

        .when(pl.col("position").is_in([
            "DE", "DT", "NT", "DL"
        ]))
        .then(pl.lit("DL"))

        .when(pl.col("position").is_in([
            "LB", "ILB", "OLB", "MLB"
        ]))
        .then(pl.lit("LB"))

        .when(pl.col("position").is_in([
            "CB", "DB"
        ]))
        .then(pl.lit("CB"))

        .when(pl.col("position").is_in([
            "S", "FS", "SS", "SAF"
        ]))
        .then(pl.lit("S"))

        .when(pl.col("position").is_in([
            "K", "P"
        ]))
        .then(pl.lit("ST"))

        .otherwise(pl.lit("OTHER"))
        .alias("projection_position")
    )
)

In [12]:
position_check = (
    player_seasons
    .group_by([
        "position",
        "projection_position"
    ])
    .agg(
        pl.len().alias("player_seasons")
    )
    .sort(
        ["projection_position", "player_seasons"],
        descending=[False, True]
    )
)

position_check

position,projection_position,player_seasons
str,str,u32
"""CB""","""CB""",1901
"""DB""","""CB""",1003
"""DE""","""DL""",1631
"""DT""","""DL""",1486
"""NT""","""DL""",148
…,…,…
"""S""","""S""",388
"""K""","""ST""",461
"""P""","""ST""",402


In [13]:
player_seasons.filter(
    pl.col("projection_position") == "OTHER"
).group_by("position").agg(
    pl.len().alias("player_seasons")
).sort(
    "player_seasons",
    descending=True
)

position,player_seasons
str,u32
"""LS""",318


## Age and Experience

Age and NFL experience provide important context for projecting future player performance.

Players at different stages of their careers can have different development, stability, and decline patterns. These variables are calculated at the player season level so they can later be shifted appropriately when constructing leakage safe future season projections.

In [14]:
print("birth_date dtype:", player_seasons.schema["birth_date"])
print("years_exp dtype:", player_seasons.schema["years_exp"])

player_seasons.select([
    "season",
    "player_display_name",
    "projection_position",
    "birth_date",
    "years_exp"
]).filter(
    pl.col("birth_date").is_not_null()
).head(10)

birth_date dtype: Date
years_exp dtype: Int32


season,player_display_name,projection_position,birth_date,years_exp
i32,str,str,date,i32
2015,"""Phil Dawson""","""ST""",1975-01-23,17
2015,"""Matt Hasselbeck""","""QB""",1975-09-25,17
2015,"""Peyton Manning""","""QB""",1976-03-24,17
2015,"""Adam Vinatieri""","""ST""",1972-12-28,19
2015,"""Charles Woodson""","""S""",1976-10-07,17
2015,"""Mike Leach""","""OTHER""",1976-10-18,15
2015,"""Shayne Graham""","""ST""",1977-12-09,15
2015,"""Tom Brady""","""QB""",1977-08-03,15
2015,"""Sebastian Janikowski""","""ST""",1978-03-02,15


In [15]:
player_seasons = (
    player_seasons
    .with_columns([
        pl.date(
            pl.col("season"),
            pl.lit(9),
            pl.lit(1)
        ).alias("season_start_date")
    ])
    .with_columns([
        (
            (
                pl.col("season_start_date") -
                pl.col("birth_date")
            ).dt.total_days() / 365.25
        ).alias("age")
    ])
    .with_columns([
        (pl.col("age") ** 2).alias("age_squared")
    ])
)

In [16]:
player_seasons = (
    player_seasons
    .with_columns(
        pl.when(pl.col("years_exp") <= 1)
        .then(pl.lit("Early Career"))

        .when(pl.col("years_exp") <= 4)
        .then(pl.lit("Developing"))

        .when(pl.col("years_exp") <= 8)
        .then(pl.lit("Prime/Veteran"))

        .otherwise(pl.lit("Veteran"))
        .alias("career_stage")
    )
)

In [17]:
player_seasons.select([
    "season",
    "player_display_name",
    "projection_position",
    "birth_date",
    "age",
    "years_exp",
    "career_stage"
]).filter(
    pl.col("birth_date").is_not_null()
).sort(
    ["season", "age"],
    descending=[True, False]
).head(20)

season,player_display_name,projection_position,birth_date,age,years_exp,career_stage
i32,str,str,date,f64,i32,str
2025,"""Dylan Sampson""","""RB""",2004-09-14,20.963723,0,"""Early Career"""
2025,"""Nic Scourton""","""LB""",2004-08-25,21.01848,0,"""Early Career"""
2025,"""LeQuint Allen Jr.""","""RB""",2004-08-05,21.073238,0,"""Early Career"""
2025,"""Harold Fannin Jr.""","""TE""",2004-07-20,21.117043,0,"""Early Career"""
2025,"""Trevor Etienne""","""RB""",2004-07-09,21.147159,0,"""Early Career"""
…,…,…,…,…,…,…
2025,"""Benjamin Morrison""","""CB""",2004-03-11,21.475702,0,"""Early Career"""
2025,"""Kelvin Banks Jr.""","""OL""",2004-03-10,21.478439,0,"""Early Career"""
2025,"""Emery Jones Jr.""","""OL""",2004-03-05,21.492129,0,"""Early Career"""


In [18]:
player_seasons.select([
    pl.col("age").min().alias("min_age"),
    pl.col("age").mean().alias("avg_age"),
    pl.col("age").max().alias("max_age"),
    pl.col("age").null_count().alias("missing_age")
])

min_age,avg_age,max_age,missing_age
f64,f64,f64,u32
20.334018,26.558226,46.67488,1


## Leakage Safe Projection Framework

Historical player statistics describe what occurred during each completed NFL season. To use these data for forecasting, each player season is converted into a future target season observation.

For a target season Y, predictor variables are restricted to information available before season Y. Historical performance is represented through lagged features from Y-1, Y-2, and Y-3, while performance during season Y is reserved as the prediction target.

This structure prevents future season information from leaking into historical model training and allows the same framework to be used for walk-forward validation and eventual 2026 projections.

In [19]:
player_seasons = (
    player_seasons
    .sort(["player_id", "season"])
)

## Historical Player Lags

Player projections use exact prior season information rather than simply taking the previous observed row.

For a target season Y:

- Lag 1 represents Y-1
- Lag 2 represents Y-2
- Lag 3 represents Y-3

If a player did not record a season in one of those years, the corresponding lag remains missing. This prevents older performance from being incorrectly treated as the immediately previous season.

In [20]:
player_history_source = (
    player_seasons
    .select([
        "season",
        "player_id",
        "player_display_name",
        "projection_position",
        "recent_team",
        "games",
        "age",
        "years_exp",
        "offense_snaps",
        "defense_snaps",
        "total_snaps",

        # Passing
        "attempts",
        "completions",
        "passing_yards",
        "passing_tds",
        "passing_interceptions",
        "sacks_suffered",
        "passing_epa",
        "passing_cpoe",

        # Rushing
        "carries",
        "rushing_yards",
        "rushing_tds",
        "rushing_epa",

        # Receiving
        "targets",
        "receptions",
        "receiving_yards",
        "receiving_tds",
        "receiving_epa",
        "target_share",
        "air_yards_share",

        # Defense
        "def_tackles_solo",
        "def_tackles_for_loss",
        "def_fumbles_forced",
        "def_sacks",
        "def_qb_hits",
        "def_interceptions",
        "def_pass_defended"
    ])
)

In [21]:
projection_rows = (
    player_seasons
    .select([
        "player_id",
        "player_display_name",
        "projection_position",
        "season",
        "age",
        "years_exp"
    ])
    .rename({
        "season": "target_season",
        "age": "target_age",
        "years_exp": "target_years_exp"
    })
)

In [22]:
def make_lag_table(df, lag):
    rename_map = {
        col: f"lag{lag}_{col}"
        for col in df.columns
        if col not in ["season", "player_id"]
    }

    return (
        df
        .with_columns(
            (pl.col("season") + lag)
            .alias("target_season")
        )
        .drop("season")
        .rename(rename_map)
    )

In [23]:
lag1 = make_lag_table(player_history_source, 1)
lag2 = make_lag_table(player_history_source, 2)
lag3 = make_lag_table(player_history_source, 3)

projection_rows = (
    projection_rows
    .join(
        lag1,
        on=["player_id", "target_season"],
        how="left"
    )
    .join(
        lag2,
        on=["player_id", "target_season"],
        how="left"
    )
    .join(
        lag3,
        on=["player_id", "target_season"],
        how="left"
    )
)

In [24]:
print(
    projection_rows
    .filter(
        pl.col("player_display_name") == "Tom Brady"
    )
    .select([
        "target_season",
        "lag1_games",
        "lag2_games",
        "lag3_games",
        "lag1_passing_yards",
        "lag2_passing_yards",
        "lag3_passing_yards"
    ])
    .sort("target_season")
)

print(
    projection_rows
    .filter(
        pl.col("player_display_name") == "Philip Rivers"
    )
    .select([
        "target_season",
        "lag1_games",
        "lag2_games",
        "lag3_games",
        "lag1_passing_yards",
        "lag2_passing_yards",
        "lag3_passing_yards"
    ])
    .sort("target_season")
)

shape: (8, 7)
┌──────────────┬────────────┬────────────┬────────────┬──────────────┬──────────────┬──────────────┐
│ target_seaso ┆ lag1_games ┆ lag2_games ┆ lag3_games ┆ lag1_passing ┆ lag2_passing ┆ lag3_passing │
│ n            ┆ ---        ┆ ---        ┆ ---        ┆ _yards       ┆ _yards       ┆ _yards       │
│ ---          ┆ i32        ┆ i32        ┆ i32        ┆ ---          ┆ ---          ┆ ---          │
│ i32          ┆            ┆            ┆            ┆ i32          ┆ i32          ┆ i32          │
╞══════════════╪════════════╪════════════╪════════════╪══════════════╪══════════════╪══════════════╡
│ 2015         ┆ null       ┆ null       ┆ null       ┆ null         ┆ null         ┆ null         │
│ 2016         ┆ 16         ┆ null       ┆ null       ┆ 4770         ┆ null         ┆ null         │
│ 2017         ┆ 12         ┆ 16         ┆ null       ┆ 3554         ┆ 4770         ┆ null         │
│ 2018         ┆ 16         ┆ 12         ┆ 16         ┆ 4577         ┆ 3554  

## Player Projection Targets

Each historical projection row is paired with the player's actual performance during the target season.

These current season values are retained strictly as modeling targets and evaluation outcomes. They are not included as predictor variables. All predictor features are derived from seasons preceding the target season or from information known before the target season began.

In [25]:
player_targets = (
    player_seasons
    .select([
        "season",
        "player_id",
        "recent_team",
        "games",

        # Playing time
        "offense_snaps",
        "defense_snaps",
        "total_snaps",

        # Passing
        "attempts",
        "passing_yards",
        "passing_tds",
        "passing_interceptions",
        "sacks_suffered",
        "passing_epa",

        # Rushing
        "carries",
        "rushing_yards",
        "rushing_tds",
        "rushing_epa",

        # Receiving
        "targets",
        "receptions",
        "receiving_yards",
        "receiving_tds",
        "receiving_epa",

        # Defense
        "def_tackles_solo",
        "def_tackles_for_loss",
        "def_fumbles_forced",
        "def_sacks",
        "def_qb_hits",
        "def_interceptions",
        "def_pass_defended"
    ])
    .rename({
        "season": "target_season",
        "recent_team": "target_team",
        "games": "target_games",

        "offense_snaps": "target_offense_snaps",
        "defense_snaps": "target_defense_snaps",
        "total_snaps": "target_total_snaps",

        "attempts": "target_attempts",
        "passing_yards": "target_passing_yards",
        "passing_tds": "target_passing_tds",
        "passing_interceptions": "target_passing_interceptions",
        "sacks_suffered": "target_sacks_suffered",
        "passing_epa": "target_passing_epa",

        "carries": "target_carries",
        "rushing_yards": "target_rushing_yards",
        "rushing_tds": "target_rushing_tds",
        "rushing_epa": "target_rushing_epa",

        "targets": "target_targets",
        "receptions": "target_receptions",
        "receiving_yards": "target_receiving_yards",
        "receiving_tds": "target_receiving_tds",
        "receiving_epa": "target_receiving_epa",

        "def_tackles_solo": "target_def_tackles_solo",
        "def_tackles_for_loss": "target_def_tackles_for_loss",
        "def_fumbles_forced": "target_def_fumbles_forced",
        "def_sacks": "target_def_sacks",
        "def_qb_hits": "target_def_qb_hits",
        "def_interceptions": "target_def_interceptions",
        "def_pass_defended": "target_def_pass_defended"
    })
)

In [26]:
projection_rows = (
    projection_rows
    .join(
        player_targets,
        on=["player_id", "target_season"],
        how="left"
    )
)

In [27]:
projection_rows = (
    projection_rows
    .with_columns([
        pl.col("lag1_games")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_lag1"),

        pl.col("lag2_games")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_lag2"),

        pl.col("lag3_games")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_lag3")
    ])
    .with_columns(
        (
            pl.col("has_lag1")
            + pl.col("has_lag2")
            + pl.col("has_lag3")
        ).alias("prior_seasons_available")
    )
)

In [28]:
projection_rows = (
    projection_rows
    .with_columns(
        (
            (pl.col("has_lag1") == 0)
            &
            (
                (pl.col("has_lag2") == 1)
                |
                (pl.col("has_lag3") == 1)
            )
        )
        .cast(pl.Int8)
        .alias("returning_after_missed_season")
    )
)

In [29]:
print("Projection rows:", projection_rows.height)

print(
    "Duplicate player target-seasons:",
    projection_rows
    .group_by(["player_id", "target_season"])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(
    projection_rows
    .group_by("target_season")
    .agg([
        pl.len().alias("players"),

        pl.col("has_lag1")
        .sum()
        .alias("with_lag1"),

        pl.col("has_lag2")
        .sum()
        .alias("with_lag2"),

        pl.col("has_lag3")
        .sum()
        .alias("with_lag3"),

        pl.col("returning_after_missed_season")
        .sum()
        .alias("returning_after_missed_season")
    ])
    .sort("target_season")
)

Projection rows: 21366
Duplicate player target-seasons: 0
shape: (11, 6)
┌───────────────┬─────────┬───────────┬───────────┬───────────┬───────────────────────────────┐
│ target_season ┆ players ┆ with_lag1 ┆ with_lag2 ┆ with_lag3 ┆ returning_after_missed_season │
│ ---           ┆ ---     ┆ ---       ┆ ---       ┆ ---       ┆ ---                           │
│ i32           ┆ u32     ┆ i64       ┆ i64       ┆ i64       ┆ i64                           │
╞═══════════════╪═════════╪═══════════╪═══════════╪═══════════╪═══════════════════════════════╡
│ 2015          ┆ 1845    ┆ 0         ┆ 0         ┆ 0         ┆ 0                             │
│ 2016          ┆ 1855    ┆ 1368      ┆ 0         ┆ 0         ┆ 0                             │
│ 2017          ┆ 1868    ┆ 1396      ┆ 1117      ┆ 0         ┆ 63                            │
│ 2018          ┆ 1883    ┆ 1402      ┆ 1158      ┆ 906       ┆ 90                            │
│ 2019          ┆ 1888    ┆ 1400      ┆ 1118      ┆ 895       ┆

## Recent Performance

Recent performance is generally more informative for projection than older performance, but relying on only one season can make projections too sensitive to short term variance.

I combine up to three prior seasons with greater weight on the most recent year. Missing seasons are not treated as zero production; the available weights are rescaled while separate history availability features preserve information about gaps in a player's career.

In [30]:
RECENCY_WEIGHTS = {
    1: 0.55,
    2: 0.30,
    3: 0.15
}


def weighted_history_expr(feature):
    numerator = (
        pl.col(f"lag1_{feature}").fill_null(0) * RECENCY_WEIGHTS[1]
        + pl.col(f"lag2_{feature}").fill_null(0) * RECENCY_WEIGHTS[2]
        + pl.col(f"lag3_{feature}").fill_null(0) * RECENCY_WEIGHTS[3]
    )

    denominator = (
        pl.col(f"lag1_{feature}").is_not_null().cast(pl.Float64)
        * RECENCY_WEIGHTS[1]
        + pl.col(f"lag2_{feature}").is_not_null().cast(pl.Float64)
        * RECENCY_WEIGHTS[2]
        + pl.col(f"lag3_{feature}").is_not_null().cast(pl.Float64)
        * RECENCY_WEIGHTS[3]
    )

    return (
        pl.when(denominator > 0)
        .then(numerator / denominator)
        .otherwise(None)
        .alias(f"weighted_{feature}")
    )

In [31]:
weighted_features = [
    # Availability / playing time
    "games",
    "offense_snaps",
    "defense_snaps",
    "total_snaps",

    # Passing
    "attempts",
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "passing_epa",
    "passing_cpoe",

    # Rushing
    "carries",
    "rushing_yards",
    "rushing_tds",
    "rushing_epa",

    # Receiving
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "receiving_epa",
    "target_share",
    "air_yards_share",

    # Defense
    "def_tackles_solo",
    "def_tackles_for_loss",
    "def_fumbles_forced",
    "def_sacks",
    "def_qb_hits",
    "def_interceptions",
    "def_pass_defended"
]

projection_rows = (
    projection_rows
    .with_columns([
        weighted_history_expr(feature)
        for feature in weighted_features
    ])
)

In [32]:
projection_rows = (
    projection_rows
    .with_columns([
        (
            pl.col("lag1_total_snaps")
            - pl.col("lag2_total_snaps")
        ).alias("recent_snap_change"),

        (
            pl.col("lag1_passing_epa")
            - pl.col("lag2_passing_epa")
        ).alias("recent_passing_epa_change"),

        (
            pl.col("lag1_rushing_epa")
            - pl.col("lag2_rushing_epa")
        ).alias("recent_rushing_epa_change"),

        (
            pl.col("lag1_receiving_epa")
            - pl.col("lag2_receiving_epa")
        ).alias("recent_receiving_epa_change")
    ])
)

## Quarterback Projection Features

Quarterback performance has an outsized effect on overall team strength, so quarterbacks receive a dedicated projection feature set.

The QB features combine recent passing efficiency, volume, accuracy, turnover tendency, rushing production, playing time, and career context. All historical inputs are lagged so that only information available before the target season is used.

In [33]:
qb_projection_rows = (
    projection_rows
    .filter(
        pl.col("projection_position") == "QB"
    )
)

In [34]:
qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        pl.when(pl.col("lag1_attempts") > 0)
        .then(
            pl.col("lag1_passing_yards")
            / pl.col("lag1_attempts")
        )
        .otherwise(None)
        .alias("lag1_yards_per_attempt"),

        pl.when(pl.col("lag2_attempts") > 0)
        .then(
            pl.col("lag2_passing_yards")
            / pl.col("lag2_attempts")
        )
        .otherwise(None)
        .alias("lag2_yards_per_attempt"),

        pl.when(pl.col("lag3_attempts") > 0)
        .then(
            pl.col("lag3_passing_yards")
            / pl.col("lag3_attempts")
        )
        .otherwise(None)
        .alias("lag3_yards_per_attempt"),

        pl.when(pl.col("lag1_attempts") > 0)
        .then(
            pl.col("lag1_passing_tds")
            / pl.col("lag1_attempts")
        )
        .otherwise(None)
        .alias("lag1_td_rate"),

        pl.when(pl.col("lag2_attempts") > 0)
        .then(
            pl.col("lag2_passing_tds")
            / pl.col("lag2_attempts")
        )
        .otherwise(None)
        .alias("lag2_td_rate"),

        pl.when(pl.col("lag3_attempts") > 0)
        .then(
            pl.col("lag3_passing_tds")
            / pl.col("lag3_attempts")
        )
        .otherwise(None)
        .alias("lag3_td_rate"),

        pl.when(pl.col("lag1_attempts") > 0)
        .then(
            pl.col("lag1_passing_interceptions")
            / pl.col("lag1_attempts")
        )
        .otherwise(None)
        .alias("lag1_int_rate"),

        pl.when(pl.col("lag2_attempts") > 0)
        .then(
            pl.col("lag2_passing_interceptions")
            / pl.col("lag2_attempts")
        )
        .otherwise(None)
        .alias("lag2_int_rate"),

        pl.when(pl.col("lag3_attempts") > 0)
        .then(
            pl.col("lag3_passing_interceptions")
            / pl.col("lag3_attempts")
        )
        .otherwise(None)
        .alias("lag3_int_rate")
    ])
)

In [35]:
qb_rate_features = [
    "yards_per_attempt",
    "td_rate",
    "int_rate"
]

for feature in qb_rate_features:
    qb_projection_rows = (
        qb_projection_rows
        .with_columns(
            weighted_history_expr(feature)
        )
    )

In [36]:
qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        pl.when(pl.col("lag1_carries") > 0)
        .then(
            pl.col("lag1_rushing_yards")
            / pl.col("lag1_carries")
        )
        .otherwise(None)
        .alias("lag1_qb_yards_per_carry"),

        pl.when(pl.col("lag2_carries") > 0)
        .then(
            pl.col("lag2_rushing_yards")
            / pl.col("lag2_carries")
        )
        .otherwise(None)
        .alias("lag2_qb_yards_per_carry"),

        pl.when(pl.col("lag3_carries") > 0)
        .then(
            pl.col("lag3_rushing_yards")
            / pl.col("lag3_carries")
        )
        .otherwise(None)
        .alias("lag3_qb_yards_per_carry")
    ])
    .with_columns(
        weighted_history_expr(
            "qb_yards_per_carry"
        )
    )
)

In [37]:
print("QB projection rows:", qb_projection_rows.height)

print(
    qb_projection_rows
    .filter(
        (pl.col("target_season") == 2025)
        & (pl.col("prior_seasons_available") > 0)
    )
    .select([
        "player_display_name",
        "target_season",
        "target_age",
        "weighted_attempts",
        "weighted_passing_yards",
        "weighted_passing_epa",
        "weighted_passing_cpoe",
        "weighted_yards_per_attempt",
        "weighted_td_rate",
        "weighted_int_rate",
        "weighted_rushing_yards",
        "weighted_qb_yards_per_carry"
    ])
    .sort(
        "weighted_passing_epa",
        descending=True
    )
    .head(20)
)

QB projection rows: 847
shape: (20, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ player_di ┆ target_se ┆ target_ag ┆ weighted_ ┆ … ┆ weighted_ ┆ weighted_ ┆ weighted_ ┆ weighted │
│ splay_nam ┆ ason      ┆ e         ┆ attempts  ┆   ┆ td_rate   ┆ int_rate  ┆ rushing_y ┆ _qb_yard │
│ e         ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ards      ┆ s_per_ca │
│ ---       ┆ i32       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ ---       ┆ rry      │
│ str       ┆           ┆           ┆           ┆   ┆           ┆           ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Jared     ┆ 2025      ┆ 30.882957 ┆ 566.0     ┆ … ┆ 0.060042  ┆ 0.019984  ┆ 48.05     ┆ 1.454461 │
│ Goff      ┆           ┆           ┆           ┆  

### Quarterback Efficiency

Raw passing EPA captures total value, but it is influenced by playing time and attempt volume.

I therefore add per attempt and per game measures so the projection model can distinguish quarterback efficiency from opportunity.

In [38]:
# Helper for weighted historical ratios

def weighted_ratio_expr(numerator_feature, denominator_feature, output_name):
    weighted_numerator = (
        pl.col(f"lag1_{numerator_feature}").fill_null(0) * RECENCY_WEIGHTS[1]
        + pl.col(f"lag2_{numerator_feature}").fill_null(0) * RECENCY_WEIGHTS[2]
        + pl.col(f"lag3_{numerator_feature}").fill_null(0) * RECENCY_WEIGHTS[3]
    )

    weighted_denominator = (
        pl.col(f"lag1_{denominator_feature}").fill_null(0) * RECENCY_WEIGHTS[1]
        + pl.col(f"lag2_{denominator_feature}").fill_null(0) * RECENCY_WEIGHTS[2]
        + pl.col(f"lag3_{denominator_feature}").fill_null(0) * RECENCY_WEIGHTS[3]
    )

    return (
        pl.when(weighted_denominator > 0)
        .then(weighted_numerator / weighted_denominator)
        .otherwise(None)
        .alias(output_name)
    )

In [39]:
# Weighted QB efficiency features

qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        weighted_ratio_expr(
            "passing_yards",
            "attempts",
            "weighted_yards_per_attempt"
        ),

        weighted_ratio_expr(
            "passing_tds",
            "attempts",
            "weighted_td_rate"
        ),

        weighted_ratio_expr(
            "passing_interceptions",
            "attempts",
            "weighted_int_rate"
        ),

        weighted_ratio_expr(
            "passing_epa",
            "attempts",
            "weighted_passing_epa_per_attempt"
        ),

        weighted_ratio_expr(
            "rushing_yards",
            "carries",
            "weighted_qb_yards_per_carry"
        ),

        weighted_ratio_expr(
            "attempts",
            "games",
            "weighted_attempts_per_game"
        )
    ])
)

In [40]:
# Historical QB passing volume

qb_projection_rows = (
    qb_projection_rows
    .with_columns(
        (
            pl.col("lag1_attempts").fill_null(0)
            + pl.col("lag2_attempts").fill_null(0)
            + pl.col("lag3_attempts").fill_null(0)
        ).alias("prior_3yr_pass_attempts")
    )
)

In [41]:
# Target-season QB efficiency outcomes

qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        pl.when(pl.col("target_attempts") > 0)
        .then(
            pl.col("target_passing_yards")
            / pl.col("target_attempts")
        )
        .otherwise(None)
        .alias("target_yards_per_attempt"),

        pl.when(pl.col("target_attempts") > 0)
        .then(
            pl.col("target_passing_tds")
            / pl.col("target_attempts")
        )
        .otherwise(None)
        .alias("target_td_rate"),

        pl.when(pl.col("target_attempts") > 0)
        .then(
            pl.col("target_passing_interceptions")
            / pl.col("target_attempts")
        )
        .otherwise(None)
        .alias("target_int_rate"),

        pl.when(pl.col("target_attempts") > 0)
        .then(
            pl.col("target_passing_epa")
            / pl.col("target_attempts")
        )
        .otherwise(None)
        .alias("target_passing_epa_per_attempt")
    ])
)

In [42]:
qb_projection_rows.filter(
    (pl.col("target_season") == 2025)
    & (pl.col("prior_seasons_available") > 0)
).select([
    "player_display_name",
    "weighted_attempts_per_game",
    "prior_3yr_pass_attempts",
    "weighted_yards_per_attempt",
    "weighted_passing_epa_per_attempt",
    "weighted_td_rate",
    "weighted_int_rate",
    "target_attempts",
    "target_yards_per_attempt",
    "target_passing_epa_per_attempt"
]).sort(
    "weighted_passing_epa_per_attempt",
    descending=True
).head(20)

player_display_name,weighted_attempts_per_game,prior_3yr_pass_attempts,weighted_yards_per_attempt,weighted_passing_epa_per_attempt,weighted_td_rate,weighted_int_rate,target_attempts,target_yards_per_attempt,target_passing_epa_per_attempt
str,f64,i32,f64,f64,f64,f64,i32,f64,f64
"""Chris Oladokun""",0.0,0,null,null,null,null,55,4.272727,-0.547693
"""Joe Milton III""",29.0,29,8.310345,0.406576,0.034483,0.0,24,7.625,0.026543
"""Tanner McKee""",22.5,45,7.177778,0.326171,0.088889,0.0,43,6.372093,-0.057254
"""Lamar Jackson""",28.00627,1257,8.359749,0.25531,0.072308,0.011977,302,8.440397,0.060123
"""Jared Goff""",33.294118,1731,8.099205,0.233729,0.059541,0.019876,578,7.896194,0.184187
…,…,…,…,…,…,…,…,…,…
"""Jayden Daniels""",28.235294,480,7.433333,0.126781,0.052083,0.01875,188,6.712766,-0.031011
"""Jalen Hurts""",27.496795,1359,7.714652,0.120374,0.046859,0.019,454,7.101322,0.075582
"""Justin Herbert""",32.838608,1659,7.284475,0.099679,0.043172,0.010119,512,7.279297,0.031474


### Quarterback Efficiency Regression

Quarterback efficiency can be highly volatile over small samples. A player with only a handful of historical pass attempts should not receive the same confidence as an established starter with several seasons of data.

To account for this, historical quarterback efficiency is regressed toward league average performance based on the amount of prior passing volume available.

League baselines are calculated using only seasons before each target season to prevent future information from leaking into the projection features.

In [43]:
# League-average QB efficiency available before each target season

qb_league_baselines = (
    projection_rows
    .filter(pl.col("projection_position") == "QB")
    .select([
        "target_season",
        "target_attempts",
        "target_passing_yards",
        "target_passing_tds",
        "target_passing_interceptions",
        "target_passing_epa"
    ])
)

qb_baseline_rows = []

for target_season in sorted(
    qb_league_baselines["target_season"].unique().to_list()
):
    historical = qb_league_baselines.filter(
        pl.col("target_season") < target_season
    )

    attempts = historical["target_attempts"].sum()

    if attempts is None or attempts == 0:
        continue

    qb_baseline_rows.append({
        "target_season": target_season,
        "league_qb_yards_per_attempt":
            historical["target_passing_yards"].sum() / attempts,

        "league_qb_passing_epa_per_attempt":
            historical["target_passing_epa"].sum() / attempts,

        "league_qb_td_rate":
            historical["target_passing_tds"].sum() / attempts,

        "league_qb_int_rate":
            historical["target_passing_interceptions"].sum() / attempts
    })

qb_league_baselines = pl.DataFrame(qb_baseline_rows)

In [44]:
qb_league_baselines.sort("target_season")

target_season,league_qb_yards_per_attempt,league_qb_passing_epa_per_attempt,league_qb_td_rate,league_qb_int_rate
i64,f64,f64,f64,f64
2016,7.251149,0.052667,0.045952,0.023851
2017,7.201708,0.056565,0.04431,0.023263
2018,7.138836,0.039714,0.043649,0.023612
2019,7.191589,0.043588,0.044556,0.023626
2020,7.196014,0.043972,0.044533,0.023452
2021,7.2016,0.050092,0.045132,0.023186
2022,7.184852,0.048263,0.0451,0.023214
2023,7.165011,0.043872,0.044622,0.023184
2024,7.148494,0.03812,0.0442,0.023217


### Regressed Quarterback Efficiency

Historical efficiency is blended with the league average quarterback baseline using prior passing volume as the measure of confidence.

Quarterbacks with large historical samples retain most of their observed efficiency, while quarterbacks with limited passing history are pulled more strongly toward league average performance.

In [45]:
# Join leakage-safe league baselines onto QB projection rows

QB_REGRESSION_ATTEMPTS = 200

qb_projection_rows = (
    qb_projection_rows
    .join(
        qb_league_baselines,
        on="target_season",
        how="left"
    )
    .with_columns(
        (
            pl.col("prior_3yr_pass_attempts")
            / (
                pl.col("prior_3yr_pass_attempts")
                + QB_REGRESSION_ATTEMPTS
            )
        ).alias("qb_efficiency_reliability")
    )
)

In [46]:
# Regress historical QB efficiency toward the prior league baseline

qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        (
            pl.col("weighted_yards_per_attempt")
            * pl.col("qb_efficiency_reliability")
            +
            pl.col("league_qb_yards_per_attempt")
            * (1 - pl.col("qb_efficiency_reliability"))
        ).alias("regressed_yards_per_attempt"),

        (
            pl.col("weighted_passing_epa_per_attempt")
            * pl.col("qb_efficiency_reliability")
            +
            pl.col("league_qb_passing_epa_per_attempt")
            * (1 - pl.col("qb_efficiency_reliability"))
        ).alias("regressed_passing_epa_per_attempt"),

        (
            pl.col("weighted_td_rate")
            * pl.col("qb_efficiency_reliability")
            +
            pl.col("league_qb_td_rate")
            * (1 - pl.col("qb_efficiency_reliability"))
        ).alias("regressed_td_rate"),

        (
            pl.col("weighted_int_rate")
            * pl.col("qb_efficiency_reliability")
            +
            pl.col("league_qb_int_rate")
            * (1 - pl.col("qb_efficiency_reliability"))
        ).alias("regressed_int_rate")
    ])
)

In [47]:
# Use league-average efficiency when no historical passing sample exists

qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        pl.when(pl.col("prior_3yr_pass_attempts") == 0)
        .then(pl.col("league_qb_yards_per_attempt"))
        .otherwise(pl.col("regressed_yards_per_attempt"))
        .alias("regressed_yards_per_attempt"),

        pl.when(pl.col("prior_3yr_pass_attempts") == 0)
        .then(pl.col("league_qb_passing_epa_per_attempt"))
        .otherwise(pl.col("regressed_passing_epa_per_attempt"))
        .alias("regressed_passing_epa_per_attempt"),

        pl.when(pl.col("prior_3yr_pass_attempts") == 0)
        .then(pl.col("league_qb_td_rate"))
        .otherwise(pl.col("regressed_td_rate"))
        .alias("regressed_td_rate"),

        pl.when(pl.col("prior_3yr_pass_attempts") == 0)
        .then(pl.col("league_qb_int_rate"))
        .otherwise(pl.col("regressed_int_rate"))
        .alias("regressed_int_rate")
    ])
)

In [48]:
qb_projection_rows.filter(
    pl.col("target_season") == 2025
).select([
    "player_display_name",
    "prior_3yr_pass_attempts",
    "qb_efficiency_reliability",
    "weighted_yards_per_attempt",
    "regressed_yards_per_attempt",
    "weighted_passing_epa_per_attempt",
    "regressed_passing_epa_per_attempt",
    "weighted_td_rate",
    "regressed_td_rate",
    "weighted_int_rate",
    "regressed_int_rate"
]).sort(
    "regressed_passing_epa_per_attempt",
    descending=True
).head(20)

player_display_name,prior_3yr_pass_attempts,qb_efficiency_reliability,weighted_yards_per_attempt,regressed_yards_per_attempt,weighted_passing_epa_per_attempt,regressed_passing_epa_per_attempt,weighted_td_rate,regressed_td_rate,weighted_int_rate,regressed_int_rate
str,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lamar Jackson""",1257,0.862732,8.359749,8.193245,0.25531,0.225542,0.072308,0.068464,0.011977,0.013498
"""Jared Goff""",1731,0.896427,8.099205,8.000558,0.233729,0.213503,0.059541,0.057963,0.019876,0.020206
"""Josh Allen""",1629,0.890651,7.60164,7.5519,0.218012,0.198377,0.055969,0.054694,0.020595,0.020865
"""Tua Tagovailoa""",1359,0.871713,7.813722,7.72816,0.219407,0.196193,0.051179,0.050297,0.020673,0.020979
"""Brock Purdy""",1069,0.842396,8.840445,8.573514,0.223474,0.194313,0.054408,0.052816,0.025676,0.025264
…,…,…,…,…,…,…,…,…,…,…
"""Dak Prescott""",1270,0.863946,7.299695,7.278888,0.095215,0.087492,0.051601,0.050609,0.023767,0.023671
"""Joe Milton III""",29,0.126638,8.310345,7.294119,0.406576,0.08507,0.034483,0.043063,0.0,0.020142
"""Sam Darnold""",731,0.785177,7.879241,7.721888,0.090801,0.079555,0.062472,0.05857,0.02197,0.022205


### Quarterback Availability and Role

Quarterback value depends on both efficiency and expected opportunity.

I include recent games, passing volume, and snap history so the model can distinguish established starters from backups, part-time players, and quarterbacks with limited recent availability.

In [49]:
qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        (
            pl.col("lag1_games").fill_null(0)
            + pl.col("lag2_games").fill_null(0)
            + pl.col("lag3_games").fill_null(0)
        ).alias("prior_3yr_games"),

        (
            pl.col("lag1_offense_snaps").fill_null(0)
            + pl.col("lag2_offense_snaps").fill_null(0)
            + pl.col("lag3_offense_snaps").fill_null(0)
        ).alias("prior_3yr_offense_snaps"),

        pl.when(pl.col("lag1_games") > 0)
        .then(
            pl.col("lag1_offense_snaps")
            / pl.col("lag1_games")
        )
        .otherwise(None)
        .alias("lag1_offense_snaps_per_game"),

        pl.when(pl.col("lag2_games") > 0)
        .then(
            pl.col("lag2_offense_snaps")
            / pl.col("lag2_games")
        )
        .otherwise(None)
        .alias("lag2_offense_snaps_per_game"),

        pl.when(pl.col("lag3_games") > 0)
        .then(
            pl.col("lag3_offense_snaps")
            / pl.col("lag3_games")
        )
        .otherwise(None)
        .alias("lag3_offense_snaps_per_game")
    ])
)

In [50]:
qb_projection_rows = (
    qb_projection_rows
    .with_columns(
        weighted_ratio_expr(
            "offense_snaps",
            "games",
            "weighted_offense_snaps_per_game"
        )
    )
)

### Quarterback Age and Experience

Age and professional experience are retained as separate predictors so downstream models can learn nonlinear quarterback development and aging patterns from historical data.

No manual age adjustment is imposed at this stage. This avoids assuming a fixed quarterback development curve before model validation.

In [51]:
# QB age and experience features

qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        pl.col("target_age").pow(2).alias("target_age_squared"),

        pl.col("target_years_exp")
        .pow(2)
        .alias("target_years_exp_squared"),

        (
            pl.col("target_age")
            * pl.col("target_years_exp")
        ).alias("age_experience_interaction"),

        pl.when(pl.col("target_years_exp") == 0)
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("rookie_qb"),

        pl.when(pl.col("target_years_exp") <= 2)
        .then(1)
        .otherwise(0)
        .cast(pl.Int8)
        .alias("early_career_qb")
    ])
)

In [52]:
qb_projection_rows.filter(
    pl.col("target_season") == 2025
).select([
    "player_display_name",
    "target_age",
    "target_years_exp",
    "rookie_qb",
    "early_career_qb",
    "prior_seasons_available",
    "prior_3yr_games"
]).sort(
    ["target_years_exp", "player_display_name"]
).head(30)

player_display_name,target_age,target_years_exp,rookie_qb,early_career_qb,prior_seasons_available,prior_3yr_games
str,f64,i32,i8,i8,i8,i32
"""Brady Cook""",23.887748,0,1,1,0,0
"""Cam Ward""",23.271732,0,1,1,0,0
"""Dillon Gabriel""",24.676249,0,1,1,0,0
"""Jalen Milroe""",22.718686,0,1,1,0,0
"""Jaxson Dart""",22.30527,0,1,1,0,0
…,…,…,…,…,…,…
"""Tanner McKee""",25.347023,2,0,1,1,2
"""Tyson Bagent""",25.232033,2,0,1,2,9
"""Brock Purdy""",25.68104,3,0,0,3,40


### Quarterback Sack Avoidance

Quarterback efficiency is also affected by how often passing plays end in sacks.

I measure sacks relative to total dropbacks so the projection features capture a quarterback's historical sack tendency alongside passing efficiency, accuracy, turnovers, rushing value, and playing time.

In [53]:
# Weighted QB sack rate and EPA per dropback

qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        (
            pl.col("lag1_attempts").fill_null(0)
            + pl.col("lag1_sacks_suffered").fill_null(0)
        ).alias("lag1_dropbacks"),

        (
            pl.col("lag2_attempts").fill_null(0)
            + pl.col("lag2_sacks_suffered").fill_null(0)
        ).alias("lag2_dropbacks"),

        (
            pl.col("lag3_attempts").fill_null(0)
            + pl.col("lag3_sacks_suffered").fill_null(0)
        ).alias("lag3_dropbacks")
    ])
)

In [54]:
qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        weighted_ratio_expr(
            "sacks_suffered",
            "dropbacks",
            "weighted_sack_rate"
        ),

        weighted_ratio_expr(
            "passing_epa",
            "dropbacks",
            "weighted_passing_epa_per_dropback"
        )
    ])
)

In [55]:
qb_projection_rows = (
    qb_projection_rows
    .with_columns([
        (
            pl.col("target_attempts")
            + pl.col("target_sacks_suffered")
        ).alias("target_dropbacks")
    ])
    .with_columns([
        pl.when(pl.col("target_dropbacks") > 0)
        .then(
            pl.col("target_sacks_suffered")
            / pl.col("target_dropbacks")
        )
        .otherwise(None)
        .alias("target_sack_rate"),

        pl.when(pl.col("target_dropbacks") > 0)
        .then(
            pl.col("target_passing_epa")
            / pl.col("target_dropbacks")
        )
        .otherwise(None)
        .alias("target_passing_epa_per_dropback")
    ])
)

In [56]:
qb_projection_rows.filter(
    (pl.col("target_season") == 2025)
    & (pl.col("prior_3yr_pass_attempts") >= 300)
).select([
    "player_display_name",
    "prior_3yr_pass_attempts",
    "weighted_sack_rate",
    "weighted_passing_epa_per_dropback",
    "weighted_passing_cpoe",
    "regressed_passing_epa_per_attempt",
    "weighted_qb_yards_per_carry"
]).sort(
    "weighted_sack_rate"
).head(20)

player_display_name,prior_3yr_pass_attempts,weighted_sack_rate,weighted_passing_epa_per_dropback,weighted_passing_cpoe,regressed_passing_epa_per_attempt,weighted_qb_yards_per_carry
str,i32,f64,f64,f64,f64,f64
"""Josh Allen""",1629,0.036472,0.210061,2.121433,0.198377,5.218056
"""Jordan Love""",1025,0.039083,0.143021,0.579639,0.130816,4.138408
"""Cooper Rush""",494,0.040592,-0.074296,-3.430188,-0.044042,0.483117
"""Bo Nix""",567,0.040609,0.058276,0.560925,0.05493,4.673913
"""Mitchell Trubisky""",313,0.048607,0.014033,2.745612,0.02399,1.320588
…,…,…,…,…,…,…
"""Joe Flacco""",643,0.057488,-0.011838,-0.888177,-0.000458,1.950617
"""Davis Mills""",554,0.057885,-0.141388,-12.877518,-0.100068,3.539007
"""Mac Jones""",1049,0.058206,-0.068268,-1.204414,-0.054723,3.130579


In [57]:
qb_draft_info = (
    draft_picks_clean
    .filter(pl.col("position") == "QB")
    .select([
        pl.col("gsis_id").alias("player_id"),
        pl.col("season").alias("draft_season"),
        pl.col("round").alias("draft_round"),
        pl.col("pick").alias("draft_pick")
    ])
    .unique(subset=["player_id"])
)

draft_columns = [
    "draft_season",
    "draft_round",
    "draft_pick",
    "top_10_pick",
    "first_round_qb",
    "drafted_this_season",
    "log_draft_capital",
    "rookie_first_round_qb",
    "rookie_top_10_pick"
]

columns_to_drop = [
    column
    for column in draft_columns
    if column in qb_projection_rows.columns
]

if columns_to_drop:
    qb_projection_rows = qb_projection_rows.drop(columns_to_drop)

qb_projection_rows = (
    qb_projection_rows
    .join(
        qb_draft_info,
        on="player_id",
        how="left"
    )
    .with_columns([
        (pl.col("draft_pick") <= 10)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("top_10_pick"),

        (pl.col("draft_pick") <= 32)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("first_round_qb"),

        (pl.col("draft_season") == pl.col("target_season"))
        .fill_null(False)
        .cast(pl.Int8)
        .alias("drafted_this_season")
    ])
)

In [58]:
qb_projection_rows = qb_projection_rows.with_columns([
    pl.when(pl.col("draft_pick").is_not_null())
    .then(
        1 - (
            pl.col("draft_pick").cast(pl.Float64).log()
            / np.log(300)
        )
    )
    .otherwise(0.0)
    .alias("log_draft_capital"),

    (
        pl.col("rookie_qb")
        * pl.col("first_round_qb")
    ).alias("rookie_first_round_qb"),

    (
        pl.col("rookie_qb")
        * pl.col("top_10_pick")
    ).alias("rookie_top_10_pick")
])

In [59]:
qb_model_features = [
    "regressed_yards_per_attempt",
    "regressed_passing_epa_per_attempt",
    "regressed_td_rate",
    "regressed_int_rate",
    "weighted_passing_epa_per_dropback",
    "weighted_sack_rate",
    "weighted_passing_cpoe",
    "weighted_qb_yards_per_carry",
    "weighted_attempts_per_game",
    "weighted_offense_snaps_per_game",
    "prior_3yr_pass_attempts",
    "prior_3yr_games",
    "target_age",
    "target_age_squared",
    "target_years_exp",
    "target_years_exp_squared",
    "age_experience_interaction",
    "rookie_qb",
    "early_career_qb",
    "returning_after_missed_season",
    "log_draft_capital",
    "rookie_first_round_qb",
    "rookie_top_10_pick"
]

qb_model_target = "target_passing_epa_per_dropback"

In [60]:
qb_model_data = (
    qb_projection_rows
    .select(
        [
            "player_id",
            "player_display_name",
            "target_season"
        ]
        + qb_model_features
        + [qb_model_target]
    )
    .filter(
        pl.col(qb_model_target).is_not_null()
    )
)

print("QB modeling rows:", qb_model_data.height)
print(
    "Seasons:",
    qb_model_data["target_season"].min(),
    "-",
    qb_model_data["target_season"].max()
)

QB modeling rows: 818
Seasons: 2015 - 2025


### Quarterback Walk Forward Validation

To evaluate the quarterback projection model without using future information, each season is predicted using only earlier seasons as training data. This recreates the information structure of a true preseason projection.

In [61]:
from sklearn.linear_model import Ridge
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [62]:
qb_model_df = qb_model_data.to_pandas()

qb_model_df["target_dropbacks"] = (
    qb_projection_rows
    .select([
        "player_id",
        "target_season",
        "target_dropbacks"
    ])
    .to_pandas()
    .set_index(["player_id", "target_season"])
    .reindex(
        qb_model_df.set_index(
            ["player_id", "target_season"]
        ).index
    )["target_dropbacks"]
    .values
)

qb_model_df[
    [
        "player_display_name",
        "target_season",
        "target_dropbacks",
        qb_model_target
    ]
].head()

,player_display_name,target_season,target_dropbacks,target_passing_epa_per_dropback
0,Matt Hasselbeck,2015,272,-0.000416
1,Peyton Manning,2015,347,-0.092094
2,Tom Brady,2015,662,0.192653
3,Tom Brady,2016,447,0.327404
4,Tom Brady,2017,616,0.228270


In [63]:
QB_RIDGE_ALPHA = 5000.0

qb_walk_forward_results = []

for test_season in range(2018, 2026):

    train = qb_model_df[
        qb_model_df["target_season"] < test_season
    ].copy()

    test = qb_model_df[
        qb_model_df["target_season"] == test_season
    ].copy()

    X_train = train[qb_model_features]
    y_train = train[qb_model_target]

    X_test = test[qb_model_features]
    y_test = test[qb_model_target]

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=QB_RIDGE_ALPHA))
    ])

    train_weights = np.sqrt(
        train["target_dropbacks"].clip(lower=1)
    )

    model.fit(
        X_train,
        y_train,
        ridge__sample_weight=train_weights
    )

    predictions = model.predict(X_test)

    season_results = test[
        [
            "player_id",
            "player_display_name",
            "target_season",
            "target_dropbacks"
        ]
    ].copy()

    season_results["actual_qb_epa_per_dropback"] = y_test.values
    season_results["projected_qb_epa_per_dropback"] = predictions

    qb_walk_forward_results.append(season_results)

qb_walk_forward_results = pd.concat(
    qb_walk_forward_results,
    ignore_index=True
)

In [64]:
qb_walk_forward_results.groupby(
    "target_season"
).agg(
    qb_rows=("player_id", "size"),
    actual_avg=("actual_qb_epa_per_dropback", "mean"),
    projected_avg=("projected_qb_epa_per_dropback", "mean"),
    actual_std=("actual_qb_epa_per_dropback", "std"),
    projected_std=("projected_qb_epa_per_dropback", "std")
).round(4)

,qb_rows,actual_avg,projected_avg,actual_std,projected_std
target_season,,,,,
2018,69,-0.0933,-0.0169,0.4397,0.0738
2019,69,-0.0906,-0.0098,0.3577,0.0806
2020,79,-0.1009,-0.0234,0.3859,0.0880
2021,75,-0.0792,-0.0126,0.4465,0.0903
2022,83,-0.1263,-0.0227,0.2833,0.0901
2023,78,-0.1292,-0.0330,0.4960,0.0805
2024,76,-0.0101,-0.0348,0.4541,0.0871
2025,77,-0.0939,-0.0259,0.3768,0.0920


In [65]:
qb_baseline = (
    qb_projection_rows
    .select([
        "player_id",
        "target_season",
        "regressed_passing_epa_per_attempt"
    ])
    .to_pandas()
)

qb_walk_forward_results = qb_walk_forward_results.merge(
    qb_baseline,
    on=["player_id", "target_season"],
    how="left"
)

In [66]:
qb_evaluation = []

for season in sorted(
    qb_walk_forward_results["target_season"].unique()
):
    season_data = qb_walk_forward_results[
        qb_walk_forward_results["target_season"] == season
    ].dropna(
        subset=[
            "actual_qb_epa_per_dropback",
            "projected_qb_epa_per_dropback",
            "regressed_passing_epa_per_attempt"
        ]
    )

    actual = season_data["actual_qb_epa_per_dropback"]

    model_pred = season_data["projected_qb_epa_per_dropback"]
    baseline_pred = season_data["regressed_passing_epa_per_attempt"]

    qb_evaluation.append({
        "season": season,
        "rows": len(season_data),

        "model_mae": mean_absolute_error(
            actual,
            model_pred
        ),

        "baseline_mae": mean_absolute_error(
            actual,
            baseline_pred
        ),

        "model_rmse": np.sqrt(
            mean_squared_error(actual, model_pred)
        ),

        "baseline_rmse": np.sqrt(
            mean_squared_error(actual, baseline_pred)
        ),

        "model_correlation": actual.corr(model_pred),
        "baseline_correlation": actual.corr(baseline_pred)
    })

qb_evaluation = pd.DataFrame(qb_evaluation)

qb_evaluation.round(4)

,season,rows,model_mae,baseline_mae,model_rmse,baseline_rmse,model_correlation,baseline_correlation
0,2018,69,0.2635,0.2732,0.4330,0.4361,0.2228,0.2950
1,2019,69,0.2033,0.2224,0.3462,0.3724,0.3375,0.1897
2,2020,79,0.2158,0.2303,0.3587,0.3894,0.4772,0.3027
3,2021,75,0.2418,0.2578,0.4287,0.4452,0.3201,0.2670
4,2022,83,0.2047,0.2287,0.2885,0.3283,0.2932,0.1066
5,2023,78,0.2594,0.2848,0.4911,0.5169,0.2203,0.0723
6,2024,76,0.2426,0.2542,0.4420,0.4406,0.2069,0.2207
7,2025,77,0.2311,0.2545,0.3448,0.3699,0.5005,0.3769


In [67]:
qb_eval_complete = qb_walk_forward_results.dropna(
    subset=[
        "actual_qb_epa_per_dropback",
        "projected_qb_epa_per_dropback",
        "regressed_passing_epa_per_attempt"
    ]
).copy()

actual = qb_eval_complete["actual_qb_epa_per_dropback"]
model_pred = qb_eval_complete["projected_qb_epa_per_dropback"]
baseline_pred = qb_eval_complete["regressed_passing_epa_per_attempt"]

qb_overall_evaluation = pd.DataFrame({
    "metric": ["MAE", "RMSE", "Correlation"],
    "ridge_model": [
        mean_absolute_error(actual, model_pred),
        np.sqrt(mean_squared_error(actual, model_pred)),
        actual.corr(model_pred)
    ],
    "baseline": [
        mean_absolute_error(actual, baseline_pred),
        np.sqrt(mean_squared_error(actual, baseline_pred)),
        actual.corr(baseline_pred)
    ]
})

qb_overall_evaluation.round(4)

,metric,ridge_model,baseline
0,MAE,0.2324,0.2506
1,RMSE,0.3957,0.4155
2,Correlation,0.3103,0.2193


In [68]:
qb_workload_features = [
    "lag1_dropbacks",
    "lag2_dropbacks",
    "lag3_dropbacks",
    "lag1_offense_snaps",
    "lag2_offense_snaps",
    "lag3_offense_snaps",
    "lag1_games",
    "lag2_games",
    "lag3_games",
    "weighted_attempts_per_game",
    "weighted_offense_snaps_per_game",
    "prior_3yr_pass_attempts",
    "prior_3yr_games",
    "prior_3yr_offense_snaps",
    "target_age",
    "target_years_exp",
    "rookie_qb",
    "early_career_qb",
    "returning_after_missed_season",
    "log_draft_capital",
    "rookie_first_round_qb",
    "rookie_top_10_pick"
]

qb_workload_data = (
    qb_projection_rows
    .select(
        [
            "player_id",
            "player_display_name",
            "target_season"
        ]
        + qb_workload_features
        + ["target_dropbacks"]
    )
    .filter(
        pl.col("target_dropbacks").is_not_null()
    )
)

print(qb_workload_data.shape)

print(
    qb_workload_data
    .group_by("target_season")
    .agg([
        pl.len().alias("qb_rows"),
        pl.col("target_dropbacks").mean().alias("avg_dropbacks"),
        pl.col("target_dropbacks").max().alias("max_dropbacks")
    ])
    .sort("target_season")
)

(847, 26)
shape: (11, 4)
┌───────────────┬─────────┬───────────────┬───────────────┐
│ target_season ┆ qb_rows ┆ avg_dropbacks ┆ max_dropbacks │
│ ---           ┆ ---     ┆ ---           ┆ ---           │
│ i32           ┆ u32     ┆ f64           ┆ i32           │
╞═══════════════╪═════════╪═══════════════╪═══════════════╡
│ 2015          ┆ 74      ┆ 263.027027    ┆ 701           │
│ 2016          ┆ 71      ┆ 272.873239    ┆ 705           │
│ 2017          ┆ 73      ┆ 255.575342    ┆ 616           │
│ 2018          ┆ 72      ┆ 262.458333    ┆ 699           │
│ 2019          ┆ 71      ┆ 268.591549    ┆ 673           │
│ …             ┆ …       ┆ …             ┆ …             │
│ 2021          ┆ 81      ┆ 243.888889    ┆ 741           │
│ 2022          ┆ 83      ┆ 232.650602    ┆ 755           │
│ 2023          ┆ 81      ┆ 242.839506    ┆ 677           │
│ 2024          ┆ 78      ┆ 244.679487    ┆ 700           │
│ 2025          ┆ 81      ┆ 230.802469    ┆ 634           │
└──────────────

In [69]:
qb_workload_df = qb_workload_data.to_pandas()

QB_WORKLOAD_RIDGE_ALPHA = 10.0

qb_workload_results = []

for test_season in range(2018, 2026):

    train = qb_workload_df[
        qb_workload_df["target_season"] < test_season
    ].copy()

    test = qb_workload_df[
        qb_workload_df["target_season"] == test_season
    ].copy()

    X_train = train[qb_workload_features]
    y_train = train["target_dropbacks"]

    X_test = test[qb_workload_features]

    workload_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=QB_WORKLOAD_RIDGE_ALPHA))
    ])

    workload_model.fit(
        X_train,
        y_train
    )

    predictions = workload_model.predict(X_test)

    season_results = test[
        [
            "player_id",
            "player_display_name",
            "target_season",
            "target_dropbacks",
            "lag1_dropbacks"
        ]
    ].copy()

    season_results["projected_dropbacks"] = np.maximum(
        predictions,
        0
    )

    qb_workload_results.append(season_results)

qb_workload_results = pd.concat(
    qb_workload_results,
    ignore_index=True
)

c:\Users\efriedman\Desktop\NFL-Season-Projections\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['lag3_offense_snaps' 'lag3_games']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\Users\efriedman\Desktop\NFL-Season-Projections\.venv\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['lag3_offense_snaps' 'lag3_games']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [70]:
qb_projection_rows = qb_projection_rows.with_columns([
    (
        pl.col("lag1_carries")
        + pl.col("lag2_carries")
        + pl.col("lag3_carries")
    ).alias("prior_3yr_qb_carries"),

    (
        0.5 * pl.col("lag1_carries").fill_null(0)
        + 0.3 * pl.col("lag2_carries").fill_null(0)
        + 0.2 * pl.col("lag3_carries").fill_null(0)
    ).alias("weighted_qb_carries"),

    (
        0.5 * pl.col("lag1_rushing_yards").fill_null(0)
        + 0.3 * pl.col("lag2_rushing_yards").fill_null(0)
        + 0.2 * pl.col("lag3_rushing_yards").fill_null(0)
    ).alias("weighted_qb_rushing_yards")
])

qb_projection_rows = qb_projection_rows.with_columns([
    pl.when(pl.col("weighted_qb_carries") > 0)
    .then(
        pl.col("weighted_qb_rushing_yards")
        / pl.col("weighted_qb_carries")
    )
    .otherwise(None)
    .alias("weighted_qb_rushing_yards_per_carry")
])

In [71]:
QB_RUSHING_REGRESSION_CARRIES = 50

qb_projection_rows = qb_projection_rows.with_columns([
    pl.when(pl.col("lag1_carries") > 0)
    .then(
        pl.col("lag1_rushing_epa")
        / pl.col("lag1_carries")
    )
    .otherwise(None)
    .alias("lag1_qb_rushing_epa_per_carry"),

    (
        pl.col("lag1_carries").fill_null(0)
        / (
            pl.col("lag1_carries").fill_null(0)
            + QB_RUSHING_REGRESSION_CARRIES
        )
    ).alias("qb_rushing_reliability")
])

qb_projection_rows = qb_projection_rows.with_columns([
    (
        pl.col("lag1_qb_rushing_epa_per_carry").fill_null(0)
        * pl.col("qb_rushing_reliability")
    ).alias("regressed_qb_rushing_epa_per_carry")
])

In [72]:
qb_historical_projections = (
    qb_walk_forward_results
    .merge(
        qb_workload_results[
            [
                "player_id",
                "target_season",
                "projected_dropbacks"
            ]
        ],
        on=["player_id", "target_season"],
        how="left"
    )
)

qb_rushing_output = (
    qb_projection_rows
    .select([
        "player_id",
        "target_season",
        "lag1_carries",
        "regressed_qb_rushing_epa_per_carry"
    ])
    .to_pandas()
)

qb_rushing_output["projected_qb_rushing_carries"] = (
    qb_rushing_output["lag1_carries"]
    .fillna(0)
    .clip(lower=0)
)

qb_rushing_output["projected_qb_rushing_epa"] = (
    qb_rushing_output["projected_qb_rushing_carries"]
    * qb_rushing_output["regressed_qb_rushing_epa_per_carry"].fillna(0)
)

qb_historical_projections = (
    qb_historical_projections
    .merge(
        qb_rushing_output,
        on=["player_id", "target_season"],
        how="left"
    )
)

qb_historical_projections["projected_qb_rushing_carries"] = (
    qb_historical_projections["lag1_carries"]
    .fillna(0)
    .clip(lower=0)
)

qb_historical_projections["projected_passing_epa"] = (
    qb_historical_projections["projected_qb_epa_per_dropback"]
    * qb_historical_projections["projected_dropbacks"]
)

qb_historical_projections["projected_total_qb_epa"] = (
    qb_historical_projections["projected_passing_epa"]
    + qb_historical_projections["projected_qb_rushing_epa"].fillna(0)
)

In [73]:
qb_historical_projections = (
    qb_walk_forward_results
    .merge(
        qb_workload_results[
            [
                "player_id",
                "target_season",
                "projected_dropbacks"
            ]
        ],
        on=["player_id", "target_season"],
        how="left"
    )
)

qb_rushing_output = (
    qb_projection_rows
    .select([
        "player_id",
        "target_season",
        "lag1_carries",
        "regressed_qb_rushing_epa_per_carry"
    ])
    .to_pandas()
)

qb_historical_projections = (
    qb_historical_projections
    .merge(
        qb_rushing_output,
        on=["player_id", "target_season"],
        how="left"
    )
)

qb_historical_projections["projected_qb_rushing_carries"] = (
    qb_historical_projections["lag1_carries"]
    .fillna(0)
    .clip(lower=0)
)

qb_historical_projections["projected_qb_rushing_epa"] = (
    qb_historical_projections["projected_qb_rushing_carries"]
    * qb_historical_projections[
        "regressed_qb_rushing_epa_per_carry"
    ].fillna(0)
)

qb_historical_projections["projected_passing_epa"] = (
    qb_historical_projections["projected_qb_epa_per_dropback"]
    * qb_historical_projections["projected_dropbacks"]
)

qb_historical_projections["projected_total_qb_epa"] = (
    qb_historical_projections["projected_passing_epa"]
    + qb_historical_projections["projected_qb_rushing_epa"]
)

In [74]:
qb_final_efficiency_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=QB_RIDGE_ALPHA))
])

qb_final_efficiency_weights = np.sqrt(
    qb_model_df["target_dropbacks"].clip(lower=1)
)

qb_final_efficiency_model.fit(
    qb_model_df[qb_model_features],
    qb_model_df[qb_model_target],
    ridge__sample_weight=qb_final_efficiency_weights
)

qb_final_workload_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=QB_WORKLOAD_RIDGE_ALPHA))
])

qb_final_workload_model.fit(
    qb_workload_df[qb_workload_features],
    qb_workload_df["target_dropbacks"]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](22,)","['lag1_dropbacks','lag2_dropbacks','lag3_dropbacks',..., 'log_draft_capital','rookie_first_round_qb','rookie_top_10_pick']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,22
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and

### 2026 QB Projection Inputs

With the historical QB models finalized, the next step is to construct the feature set used to generate 2026 projections. Historical performance from 2023–2025 is used to create lagged and weighted features, while league-average efficiency through 2025 provides a preseason baseline for regression.

This keeps the 2026 projection process consistent with the leakage safe framework used during historical validation.

In [75]:
qb_2026_rows = (
    player_history_source
    .filter(
        pl.col("projection_position") == "QB"
    )
    .select([
        "player_id",
        "player_display_name"
    ])
    .unique()
    .with_columns(
        pl.lit(2026).alias("target_season")
    )
)

for lag, source_season in [
    (1, 2025),
    (2, 2024),
    (3, 2023)
]:

    lag_data = (
        player_history_source
        .filter(
            pl.col("season") == source_season
        )
        .select([
            "player_id",
            "games",
            "offense_snaps",
            "attempts",
            "passing_yards",
            "passing_tds",
            "passing_interceptions",
            "passing_epa",
            "passing_cpoe",
            "carries",
            "rushing_yards",
            "rushing_epa",
            "sacks_suffered"
        ])
        .rename({
            column: f"lag{lag}_{column}"
            for column in [
                "games",
                "offense_snaps",
                "attempts",
                "passing_yards",
                "passing_tds",
                "passing_interceptions",
                "passing_epa",
                "passing_cpoe",
                "carries",
                "rushing_yards",
                "rushing_epa",
                "sacks_suffered"
            ]
        })
    )

    qb_2026_rows = qb_2026_rows.join(
        lag_data,
        on="player_id",
        how="left"
    )

In [76]:
qb_2026_age_exp = (
    player_history_source
    .filter(
        pl.col("projection_position") == "QB"
    )
    .sort(
        ["player_id", "season"]
    )
    .group_by("player_id")
    .agg([
        pl.col("player_display_name").last(),
        pl.col("age").last().alias("latest_age"),
        pl.col("years_exp").last().alias("latest_years_exp"),
        pl.col("season").last().alias("latest_season")
    ])
    .with_columns([
        (
            pl.col("latest_age")
            + (2026 - pl.col("latest_season"))
        ).alias("target_age"),

        (
            pl.col("latest_years_exp")
            + (2026 - pl.col("latest_season"))
        ).alias("target_years_exp")
    ])
    .select([
        "player_id",
        "target_age",
        "target_years_exp"
    ])
)

qb_2026_rows = (
    qb_2026_rows
    .join(
        qb_2026_age_exp,
        on="player_id",
        how="left"
    )
)

In [77]:
qb_2026_rows = (
    qb_2026_rows
    .with_columns([
        (pl.col("target_age") ** 2).alias("target_age_squared"),
        (pl.col("target_years_exp") ** 2).alias("target_years_exp_squared"),
        (
            pl.col("target_age")
            * pl.col("target_years_exp")
        ).alias("age_experience_interaction"),
        (pl.col("target_years_exp") == 0)
        .cast(pl.Int8)
        .alias("rookie_qb"),
        (pl.col("target_years_exp") <= 2)
        .cast(pl.Int8)
        .alias("early_career_qb")
    ])
)

qb_2026_draft_info = (
    draft_picks_clean
    .filter(pl.col("position") == "QB")
    .select([
        pl.col("gsis_id").alias("player_id"),
        pl.col("season").alias("draft_season"),
        pl.col("round").alias("draft_round"),
        pl.col("pick").alias("draft_pick")
    ])
    .unique(subset=["player_id"])
)

qb_2026_rows = (
    qb_2026_rows
    .join(
        qb_2026_draft_info,
        on="player_id",
        how="left"
    )
    .with_columns([
        (pl.col("draft_pick") <= 10)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("top_10_pick"),

        (pl.col("draft_pick") <= 32)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("first_round_qb"),

        (pl.col("draft_season") == 2026)
        .fill_null(False)
        .cast(pl.Int8)
        .alias("drafted_this_season")
    ])
    .with_columns([
        pl.when(pl.col("draft_pick").is_not_null())
        .then(
            1 - (
                pl.col("draft_pick").cast(pl.Float64).log()
                / np.log(300)
            )
        )
        .otherwise(0.0)
        .alias("log_draft_capital"),

        (
            pl.col("rookie_qb")
            * pl.col("first_round_qb")
        ).alias("rookie_first_round_qb"),

        (
            pl.col("rookie_qb")
            * pl.col("top_10_pick")
        ).alias("rookie_top_10_pick")
    ])
)

In [78]:
qb_2026_rows = qb_2026_rows.with_columns([
    (
        pl.col("lag1_attempts").fill_null(0)
        + pl.col("lag1_sacks_suffered").fill_null(0)
    ).alias("lag1_dropbacks"),

    (
        pl.col("lag2_attempts").fill_null(0)
        + pl.col("lag2_sacks_suffered").fill_null(0)
    ).alias("lag2_dropbacks"),

    (
        pl.col("lag3_attempts").fill_null(0)
        + pl.col("lag3_sacks_suffered").fill_null(0)
    ).alias("lag3_dropbacks"),

    (
        pl.col("lag1_attempts").fill_null(0)
        + pl.col("lag2_attempts").fill_null(0)
        + pl.col("lag3_attempts").fill_null(0)
    ).alias("prior_3yr_pass_attempts"),

    (
        pl.col("lag1_games").fill_null(0)
        + pl.col("lag2_games").fill_null(0)
        + pl.col("lag3_games").fill_null(0)
    ).alias("prior_3yr_games"),

    (
        pl.col("lag1_offense_snaps").fill_null(0)
        + pl.col("lag2_offense_snaps").fill_null(0)
        + pl.col("lag3_offense_snaps").fill_null(0)
    ).alias("prior_3yr_offense_snaps"),

    (
        pl.col("lag1_games").is_null()
        & (
            pl.col("lag2_games").is_not_null()
            | pl.col("lag3_games").is_not_null()
        )
    )
    .cast(pl.Int8)
    .alias("returning_after_missed_season")
])

In [79]:
qb_2026_rows = qb_2026_rows.with_columns([
    (
        0.5 * pl.col("lag1_attempts").fill_null(0)
        + 0.3 * pl.col("lag2_attempts").fill_null(0)
        + 0.2 * pl.col("lag3_attempts").fill_null(0)
    ).alias("weighted_attempts"),

    (
        0.5 * pl.col("lag1_games").fill_null(0)
        + 0.3 * pl.col("lag2_games").fill_null(0)
        + 0.2 * pl.col("lag3_games").fill_null(0)
    ).alias("weighted_games"),

    (
        0.5 * pl.col("lag1_offense_snaps").fill_null(0)
        + 0.3 * pl.col("lag2_offense_snaps").fill_null(0)
        + 0.2 * pl.col("lag3_offense_snaps").fill_null(0)
    ).alias("weighted_offense_snaps"),

    (
        0.5 * pl.col("lag1_passing_yards").fill_null(0)
        + 0.3 * pl.col("lag2_passing_yards").fill_null(0)
        + 0.2 * pl.col("lag3_passing_yards").fill_null(0)
    ).alias("weighted_passing_yards"),

    (
        0.5 * pl.col("lag1_passing_tds").fill_null(0)
        + 0.3 * pl.col("lag2_passing_tds").fill_null(0)
        + 0.2 * pl.col("lag3_passing_tds").fill_null(0)
    ).alias("weighted_passing_tds"),

    (
        0.5 * pl.col("lag1_passing_interceptions").fill_null(0)
        + 0.3 * pl.col("lag2_passing_interceptions").fill_null(0)
        + 0.2 * pl.col("lag3_passing_interceptions").fill_null(0)
    ).alias("weighted_passing_interceptions"),

    (
        0.5 * pl.col("lag1_passing_epa").fill_null(0)
        + 0.3 * pl.col("lag2_passing_epa").fill_null(0)
        + 0.2 * pl.col("lag3_passing_epa").fill_null(0)
    ).alias("weighted_passing_epa"),

    (
        0.5 * pl.col("lag1_sacks_suffered").fill_null(0)
        + 0.3 * pl.col("lag2_sacks_suffered").fill_null(0)
        + 0.2 * pl.col("lag3_sacks_suffered").fill_null(0)
    ).alias("weighted_sacks_suffered"),

    (
        0.5 * pl.col("lag1_dropbacks").fill_null(0)
        + 0.3 * pl.col("lag2_dropbacks").fill_null(0)
        + 0.2 * pl.col("lag3_dropbacks").fill_null(0)
    ).alias("weighted_dropbacks")
])

In [80]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.when(pl.col("weighted_games") > 0)
    .then(
        pl.col("weighted_attempts")
        / pl.col("weighted_games")
    )
    .otherwise(None)
    .alias("weighted_attempts_per_game"),

    pl.when(pl.col("weighted_games") > 0)
    .then(
        pl.col("weighted_offense_snaps")
        / pl.col("weighted_games")
    )
    .otherwise(None)
    .alias("weighted_offense_snaps_per_game"),

    pl.when(pl.col("weighted_attempts") > 0)
    .then(
        pl.col("weighted_passing_yards")
        / pl.col("weighted_attempts")
    )
    .otherwise(None)
    .alias("weighted_yards_per_attempt"),

    pl.when(pl.col("weighted_attempts") > 0)
    .then(
        pl.col("weighted_passing_tds")
        / pl.col("weighted_attempts")
    )
    .otherwise(None)
    .alias("weighted_td_rate"),

    pl.when(pl.col("weighted_attempts") > 0)
    .then(
        pl.col("weighted_passing_interceptions")
        / pl.col("weighted_attempts")
    )
    .otherwise(None)
    .alias("weighted_int_rate"),

    pl.when(pl.col("weighted_attempts") > 0)
    .then(
        pl.col("weighted_passing_epa")
        / pl.col("weighted_attempts")
    )
    .otherwise(None)
    .alias("weighted_passing_epa_per_attempt"),

    pl.when(pl.col("weighted_dropbacks") > 0)
    .then(
        pl.col("weighted_passing_epa")
        / pl.col("weighted_dropbacks")
    )
    .otherwise(None)
    .alias("weighted_passing_epa_per_dropback"),

    pl.when(pl.col("weighted_dropbacks") > 0)
    .then(
        pl.col("weighted_sacks_suffered")
        / pl.col("weighted_dropbacks")
    )
    .otherwise(None)
    .alias("weighted_sack_rate")
])

In [81]:
qb_2026_rows = qb_2026_rows.with_columns([
    (
        0.5 * pl.col("lag1_passing_cpoe").fill_null(0)
        + 0.3 * pl.col("lag2_passing_cpoe").fill_null(0)
        + 0.2 * pl.col("lag3_passing_cpoe").fill_null(0)
    ).alias("weighted_passing_cpoe"),

    (
        0.5 * pl.col("lag1_carries").fill_null(0)
        + 0.3 * pl.col("lag2_carries").fill_null(0)
        + 0.2 * pl.col("lag3_carries").fill_null(0)
    ).alias("weighted_qb_carries"),

    (
        0.5 * pl.col("lag1_rushing_yards").fill_null(0)
        + 0.3 * pl.col("lag2_rushing_yards").fill_null(0)
        + 0.2 * pl.col("lag3_rushing_yards").fill_null(0)
    ).alias("weighted_qb_rushing_yards")
])

qb_2026_rows = qb_2026_rows.with_columns([
    pl.when(pl.col("weighted_qb_carries") > 0)
    .then(
        pl.col("weighted_qb_rushing_yards")
        / pl.col("weighted_qb_carries")
    )
    .otherwise(None)
    .alias("weighted_qb_yards_per_carry")
])

In [82]:
qb_2025_league = (
    player_history_source
    .filter(
        (pl.col("season") == 2025)
        & (pl.col("projection_position") == "QB")
    )
)

league_2026_attempts = (
    qb_2025_league["attempts"]
    .fill_null(0)
    .sum()
)

league_2026_yards = (
    qb_2025_league["passing_yards"]
    .fill_null(0)
    .sum()
)

league_2026_passing_epa = (
    qb_2025_league["passing_epa"]
    .fill_null(0)
    .sum()
)

league_2026_tds = (
    qb_2025_league["passing_tds"]
    .fill_null(0)
    .sum()
)

league_2026_interceptions = (
    qb_2025_league["passing_interceptions"]
    .fill_null(0)
    .sum()
)

league_2026_yards_per_attempt = (
    league_2026_yards
    / league_2026_attempts
)

league_2026_passing_epa_per_attempt = (
    league_2026_passing_epa
    / league_2026_attempts
)

league_2026_td_rate = (
    league_2026_tds
    / league_2026_attempts
)

league_2026_int_rate = (
    league_2026_interceptions
    / league_2026_attempts
)

In [83]:
qb_2026_rows = qb_2026_rows.with_columns([
    (
        pl.col("prior_3yr_pass_attempts")
        / (
            pl.col("prior_3yr_pass_attempts")
            + QB_REGRESSION_ATTEMPTS
        )
    ).alias("qb_efficiency_reliability")
])

qb_2026_rows = qb_2026_rows.with_columns([
    (
        pl.col("qb_efficiency_reliability")
        * pl.col("weighted_yards_per_attempt").fill_null(
            league_2026_yards_per_attempt
        )
        + (
            1 - pl.col("qb_efficiency_reliability")
        ) * league_2026_yards_per_attempt
    ).alias("regressed_yards_per_attempt"),

    (
        pl.col("qb_efficiency_reliability")
        * pl.col("weighted_passing_epa_per_attempt").fill_null(
            league_2026_passing_epa_per_attempt
        )
        + (
            1 - pl.col("qb_efficiency_reliability")
        ) * league_2026_passing_epa_per_attempt
    ).alias("regressed_passing_epa_per_attempt"),

    (
        pl.col("qb_efficiency_reliability")
        * pl.col("weighted_td_rate").fill_null(
            league_2026_td_rate
        )
        + (
            1 - pl.col("qb_efficiency_reliability")
        ) * league_2026_td_rate
    ).alias("regressed_td_rate"),

    (
        pl.col("qb_efficiency_reliability")
        * pl.col("weighted_int_rate").fill_null(
            league_2026_int_rate
        )
        + (
            1 - pl.col("qb_efficiency_reliability")
        ) * league_2026_int_rate
    ).alias("regressed_int_rate")
])

In [84]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.when(pl.col("lag1_carries") > 0)
    .then(
        pl.col("lag1_rushing_epa")
        / pl.col("lag1_carries")
    )
    .otherwise(None)
    .alias("lag1_qb_rushing_epa_per_carry"),

    (
        pl.col("lag1_carries").fill_null(0)
        / (
            pl.col("lag1_carries").fill_null(0)
            + QB_RUSHING_REGRESSION_CARRIES
        )
    ).alias("qb_rushing_reliability")
])

qb_2026_rows = qb_2026_rows.with_columns([
    (
        pl.col("lag1_qb_rushing_epa_per_carry").fill_null(0)
        * pl.col("qb_rushing_reliability")
    ).alias("regressed_qb_rushing_epa_per_carry")
])

In [85]:
qb_2026_df = qb_2026_rows.to_pandas()

qb_2026_df["projected_qb_epa_per_dropback"] = (
    qb_final_efficiency_model.predict(
        qb_2026_df[qb_model_features]
    )
)

qb_2026_df["projected_dropbacks"] = np.maximum(
    qb_final_workload_model.predict(
        qb_2026_df[qb_workload_features]
    ),
    0
)

### Generating 2026 QB Projections

The finalized passing efficiency and workload models are applied to the 2026 feature set. Passing efficiency and expected workload are projected separately, while rushing value is estimated using prior rushing volume and regressed rushing EPA per carry.

These components are then combined to estimate each quarterback's total projected contribution.

In [86]:
qb_2026_df = qb_2026_rows.to_pandas()

qb_2026_df["projected_qb_epa_per_dropback"] = (
    qb_final_efficiency_model.predict(
        qb_2026_df[qb_model_features]
    )
)

qb_2026_df["projected_dropbacks"] = np.maximum(
    qb_final_workload_model.predict(
        qb_2026_df[qb_workload_features]
    ),
    0
)

In [87]:
qb_2026_df["projected_qb_rushing_carries"] = (
    qb_2026_df["lag1_carries"]
    .fillna(0)
    .clip(lower=0)
)

qb_2026_df["projected_qb_rushing_epa"] = (
    qb_2026_df["projected_qb_rushing_carries"]
    * qb_2026_df[
        "regressed_qb_rushing_epa_per_carry"
    ].fillna(0)
)

In [88]:
qb_2026_df["projected_passing_epa"] = (
    qb_2026_df["projected_qb_epa_per_dropback"]
    * qb_2026_df["projected_dropbacks"]
)

qb_2026_df["projected_total_qb_epa"] = (
    qb_2026_df["projected_passing_epa"]
    + qb_2026_df["projected_qb_rushing_epa"]
)

In [89]:
qb_2026_projections = (
    qb_2026_df[
        [
            "player_id",
            "player_display_name",
            "target_age",
            "target_years_exp",
            "projected_dropbacks",
            "projected_qb_epa_per_dropback",
            "projected_passing_epa",
            "projected_qb_rushing_carries",
            "projected_qb_rushing_epa",
            "projected_total_qb_epa"
        ]
    ]
    .copy()
)

qb_2026_projections = (
    qb_2026_projections
    .sort_values(
        "projected_total_qb_epa",
        ascending=False
    )
    .reset_index(drop=True)
)

In [90]:
qb_2026_rows = (
    qb_2026_rows
    .filter(
        pl.col("prior_3yr_games") > 0
    )
)

In [91]:
qb_2026_df = qb_2026_rows.to_pandas()

qb_2026_df["projected_qb_epa_per_dropback"] = (
    qb_final_efficiency_model.predict(
        qb_2026_df[qb_model_features]
    )
)

qb_2026_df["projected_dropbacks"] = np.maximum(
    qb_final_workload_model.predict(
        qb_2026_df[qb_workload_features]
    ),
    0
)

In [92]:
qb_2026_df["projected_qb_rushing_carries"] = (
    qb_2026_df["lag1_carries"]
    .fillna(0)
    .clip(lower=0)
)

qb_2026_df["projected_qb_rushing_epa"] = (
    qb_2026_df["projected_qb_rushing_carries"]
    * qb_2026_df[
        "regressed_qb_rushing_epa_per_carry"
    ].fillna(0)
)

qb_2026_df["projected_passing_epa"] = (
    qb_2026_df["projected_qb_epa_per_dropback"]
    * qb_2026_df["projected_dropbacks"]
)

qb_2026_df["projected_total_qb_epa"] = (
    qb_2026_df["projected_passing_epa"]
    + qb_2026_df["projected_qb_rushing_epa"]
)

In [93]:
qb_2026_projections = (
    qb_2026_df[
        [
            "player_id",
            "player_display_name",
            "target_age",
            "target_years_exp",
            "projected_dropbacks",
            "projected_qb_epa_per_dropback",
            "projected_passing_epa",
            "projected_qb_rushing_carries",
            "projected_qb_rushing_epa",
            "projected_total_qb_epa"
        ]
    ]
    .copy()
    .sort_values(
        "projected_total_qb_epa",
        ascending=False
    )
    .reset_index(drop=True)
)

In [94]:
current_2026_rosters = nfl.load_rosters([2026])
current_2026_draft = nfl.load_draft_picks([2026])

In [95]:
qb_2026_roster = (
    current_2026_rosters
    .filter(
        (pl.col("position") == "QB")
        & pl.col("gsis_id").is_not_null()
    )
    .sort([
        "gsis_id",
        "week"
    ])
    .group_by("gsis_id")
    .agg([
        pl.col("team").last().alias("team"),
        pl.col("full_name").last().alias("player_display_name"),
        pl.col("birth_date").last().alias("birth_date"),
        pl.col("years_exp").last().alias("target_years_exp"),
        pl.col("status").last().alias("roster_status"),
        pl.col("week").last().alias("roster_week"),
        pl.col("rookie_year").last().alias("rookie_year"),
        pl.col("draft_number").last().alias("roster_draft_pick")
    ])
    .rename({
        "gsis_id": "player_id"
    })
    .with_columns(
        pl.lit(2026).alias("target_season")
    )
)

In [96]:
qb_2026_history_features = (
    qb_2026_rows
    .drop([
        "player_display_name",
        "target_season",
        "target_age",
        "target_years_exp",
        "target_age_squared",
        "target_years_exp_squared",
        "age_experience_interaction",
        "rookie_qb",
        "early_career_qb",
        "draft_season",
        "draft_round",
        "draft_pick",
        "top_10_pick",
        "first_round_qb",
        "drafted_this_season",
        "log_draft_capital",
        "rookie_first_round_qb",
        "rookie_top_10_pick"
    ])
)

In [97]:
qb_2026_rows = (
    qb_2026_roster
    .join(
        qb_2026_history_features,
        on="player_id",
        how="left"
    )
)

In [98]:
qb_2026_draft = (
    current_2026_draft
    .filter(
        (pl.col("position") == "QB")
        & pl.col("gsis_id").is_not_null()
    )
    .select([
        pl.col("gsis_id").alias("player_id"),
        pl.col("season").alias("draft_season"),
        pl.col("round").alias("draft_round"),
        pl.col("pick").alias("draft_pick")
    ])
    .unique(
        subset=["player_id"]
    )
)

qb_2026_rows = (
    qb_2026_rows
    .join(
        qb_2026_draft,
        on="player_id",
        how="left"
    )
)

In [99]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.coalesce([
        pl.col("draft_pick"),
        pl.col("roster_draft_pick")
    ]).alias("draft_pick"),

    pl.when(
        pl.col("draft_season").is_not_null()
    )
    .then(
        pl.col("draft_season")
    )
    .otherwise(
        pl.col("rookie_year")
    )
    .alias("draft_season")
])

In [100]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.col("birth_date")
    .cast(pl.Date, strict=False)
    .alias("birth_date")
])

qb_2026_rows = qb_2026_rows.with_columns([
    (
        (
            pl.date(2026, 9, 1)
            - pl.col("birth_date")
        ).dt.total_days()
        / 365.2425
    ).alias("target_age")
])

In [101]:
qb_2026_rows = qb_2026_rows.with_columns([
    (pl.col("target_age") ** 2).alias("target_age_squared"),

    (
        pl.col("target_years_exp") ** 2
    ).alias("target_years_exp_squared"),

    (
        pl.col("target_age")
        * pl.col("target_years_exp")
    ).alias("age_experience_interaction"),

    (
        pl.col("target_years_exp") == 0
    )
    .fill_null(False)
    .cast(pl.Int8)
    .alias("rookie_qb"),

    (
        pl.col("target_years_exp") <= 2
    )
    .fill_null(False)
    .cast(pl.Int8)
    .alias("early_career_qb"),

    (
        pl.col("draft_pick") <= 10
    )
    .fill_null(False)
    .cast(pl.Int8)
    .alias("top_10_pick"),

    (
        pl.col("draft_pick") <= 32
    )
    .fill_null(False)
    .cast(pl.Int8)
    .alias("first_round_qb"),

    (
        pl.col("draft_season") == 2026
    )
    .fill_null(False)
    .cast(pl.Int8)
    .alias("drafted_this_season")
])

In [102]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.when(
        pl.col("draft_pick").is_not_null()
    )
    .then(
        1 - (
            pl.col("draft_pick").cast(pl.Float64).log()
            / np.log(300)
        )
    )
    .otherwise(0.0)
    .alias("log_draft_capital"),

    (
        pl.col("rookie_qb")
        * pl.col("first_round_qb")
    ).alias("rookie_first_round_qb"),

    (
        pl.col("rookie_qb")
        * pl.col("top_10_pick")
    ).alias("rookie_top_10_pick")
])

In [103]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.col("prior_3yr_pass_attempts")
    .fill_null(0)
    .alias("prior_3yr_pass_attempts"),

    pl.col("prior_3yr_games")
    .fill_null(0)
    .alias("prior_3yr_games"),

    pl.col("prior_3yr_offense_snaps")
    .fill_null(0)
    .alias("prior_3yr_offense_snaps"),

    pl.col("returning_after_missed_season")
    .fill_null(0)
    .alias("returning_after_missed_season"),

    pl.col("regressed_yards_per_attempt")
    .fill_null(
        league_2026_yards_per_attempt
    )
    .alias("regressed_yards_per_attempt"),

    pl.col("regressed_passing_epa_per_attempt")
    .fill_null(
        league_2026_passing_epa_per_attempt
    )
    .alias("regressed_passing_epa_per_attempt"),

    pl.col("regressed_td_rate")
    .fill_null(
        league_2026_td_rate
    )
    .alias("regressed_td_rate"),

    pl.col("regressed_int_rate")
    .fill_null(
        league_2026_int_rate
    )
    .alias("regressed_int_rate")
])

In [104]:
qb_2026_rows = qb_2026_rows.with_columns([
    pl.col("lag1_carries")
    .fill_null(0)
    .alias("lag1_carries"),

    pl.col("regressed_qb_rushing_epa_per_carry")
    .fill_null(0)
    .alias("regressed_qb_rushing_epa_per_carry")
])

In [105]:
qb_2026_df = qb_2026_rows.to_pandas()

qb_2026_df["projected_qb_epa_per_dropback"] = (
    qb_final_efficiency_model.predict(
        qb_2026_df[qb_model_features]
    )
)

qb_2026_df["projected_dropbacks"] = np.maximum(
    qb_final_workload_model.predict(
        qb_2026_df[qb_workload_features]
    ),
    0
)

In [106]:
qb_2026_df["projected_qb_rushing_carries"] = (
    qb_2026_df["lag1_carries"]
    .fillna(0)
    .clip(lower=0)
)

qb_2026_df["projected_qb_rushing_epa"] = (
    qb_2026_df["projected_qb_rushing_carries"]
    * qb_2026_df[
        "regressed_qb_rushing_epa_per_carry"
    ].fillna(0)
)

qb_2026_df["projected_passing_epa"] = (
    qb_2026_df["projected_qb_epa_per_dropback"]
    * qb_2026_df["projected_dropbacks"]
)

qb_2026_df["projected_total_qb_epa"] = (
    qb_2026_df["projected_passing_epa"]
    + qb_2026_df["projected_qb_rushing_epa"]
)

In [107]:
qb_2026_projections = (
    qb_2026_df[
        [
            "player_id",
            "player_display_name",
            "team",
            "roster_status",
            "roster_week",
            "target_age",
            "target_years_exp",
            "rookie_qb",
            "draft_pick",
            "projected_dropbacks",
            "projected_qb_epa_per_dropback",
            "projected_passing_epa",
            "projected_qb_rushing_carries",
            "projected_qb_rushing_epa",
            "projected_total_qb_epa"
        ]
    ]
    .copy()
    .sort_values(
        "projected_total_qb_epa",
        ascending=False
    )
    .reset_index(drop=True)
)

In [108]:
qb_2026_projections["team_projected_qb_dropbacks"] = (
    qb_2026_projections
    .groupby("team")["projected_dropbacks"]
    .transform("sum")
)

qb_2026_projections["projected_dropback_share"] = np.where(
    qb_2026_projections["team_projected_qb_dropbacks"] > 0,
    qb_2026_projections["projected_dropbacks"]
    / qb_2026_projections["team_projected_qb_dropbacks"],
    0
)

In [109]:
qb_2026_projections["projected_qb_room_rank"] = (
    qb_2026_projections
    .groupby("team")["projected_dropbacks"]
    .rank(
        method="first",
        ascending=False
    )
    .astype(int)
)

In [110]:
qb_2026_team_qb1 = (
    qb_2026_projections[
        qb_2026_projections["projected_qb_room_rank"] == 1
    ]
    .copy()
    .sort_values(
        "projected_total_qb_epa",
        ascending=False
    )
    .reset_index(drop=True)
)

In [111]:
current_2026_depth_charts = nfl.load_depth_charts([2026])

In [112]:
# TEMPORARY / DELETE AFTER CHECK — Inspect 2026 QB depth-chart ranks

qb_depth_check = (
    current_2026_depth_charts
    .filter(
        pl.col("pos_abb") == "QB"
    )
    .select([
        "dt",
        "team",
        "player_name",
        "gsis_id",
        "pos_grp",
        "pos_name",
        "pos_abb",
        "pos_slot",
        "pos_rank"
    ])
    .sort([
        "team",
        "dt",
        "pos_rank"
    ])
)

print("QB depth-chart rows:", qb_depth_check.height)

print("\nLatest depth-chart date:")
print(
    qb_depth_check["dt"].max()
)

print("\nLatest QB depth chart:")
print(
    qb_depth_check
    .filter(
        pl.col("dt") == qb_depth_check["dt"].max()
    )
    .sort([
        "team",
        "pos_rank"
    ])
)

QB depth-chart rows: 18084

Latest depth-chart date:
2026-08-28T19:13:13Z

Latest QB depth chart:
shape: (121, 9)
┌──────────────┬──────┬─────────────┬────────────┬───┬─────────────┬─────────┬──────────┬──────────┐
│ dt           ┆ team ┆ player_name ┆ gsis_id    ┆ … ┆ pos_name    ┆ pos_abb ┆ pos_slot ┆ pos_rank │
│ ---          ┆ ---  ┆ ---         ┆ ---        ┆   ┆ ---         ┆ ---     ┆ ---      ┆ ---      │
│ str          ┆ str  ┆ str         ┆ str        ┆   ┆ str         ┆ str     ┆ i32      ┆ i32      │
╞══════════════╪══════╪═════════════╪════════════╪═══╪═════════════╪═════════╪══════════╪══════════╡
│ 2026-08-28T1 ┆ ARI  ┆ Jacoby      ┆ 00-0033119 ┆ … ┆ Quarterback ┆ QB      ┆ 9        ┆ 1        │
│ 9:13:13Z     ┆      ┆ Brissett    ┆            ┆   ┆             ┆         ┆          ┆          │
│ 2026-08-28T1 ┆ ARI  ┆ Gardner     ┆ 00-0035289 ┆ … ┆ Quarterback ┆ QB      ┆ 9        ┆ 2        │
│ 9:13:13Z     ┆      ┆ Minshew II  ┆            ┆   ┆             ┆         ┆